# 03 — Comparator Source Recovery and Full-Text Extraction

Purpose: take the focused comparator review list from notebook `02`, recover public PDFs/HTML where possible, extract full text, and create a manual-download queue for blocked or paywalled sources.

This notebook keeps comparator sources separate from the data-centre sources. It does not overwrite the data-centre `raw_sources` or `fulltext` folders.

In [ ]:
# Optional first-time setup:
# %pip install -q pandas requests beautifulsoup4 pdfplumber pypdf openpyxl

from pathlib import Path
from urllib.parse import urlparse, urljoin
import hashlib
import math
import re
import time
import pandas as pd
import requests

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "Research_Workflow").is_dir() and (p.name == "DC_job_potential" or (p / "Media_Article").exists()):
            return p
    candidate = Path.cwd().resolve() / "DC_job_potential"
    if (candidate / "Research_Workflow").is_dir():
        return candidate
    raise RuntimeError("Could not locate DC_job_potential root. Run inside the project folder.")


ROOT = find_project_root(Path.cwd())
WORKFLOW = ROOT / "Research_Workflow"
EXTRACTION = WORKFLOW / "02_Source_Extraction"
DATA_DIR = EXTRACTION / "data"
COMPARATOR_DIR = DATA_DIR / "comparator_sources"
RAW_DIR = COMPARATOR_DIR / "raw_sources"
TEXT_DIR = COMPARATOR_DIR / "fulltext"
QUEUE_DIR = DATA_DIR / "manual_RA_queue"
REVIEW_DIR = DATA_DIR / "coding_outputs" / "comparator_review"

for d in (RAW_DIR, TEXT_DIR, QUEUE_DIR, REVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)

REVIEW_TEMPLATE = REVIEW_DIR / "comparator_metric_review_template.csv"

print("ROOT:", ROOT)
print("REVIEW_TEMPLATE:", REVIEW_TEMPLATE)
print("RAW_DIR:", RAW_DIR)
print("TEXT_DIR:", TEXT_DIR)

In [ ]:
# Controls
RUN_DOWNLOADS = True
REQUEST_TIMEOUT = 40
SLEEP_BETWEEN_DOWNLOADS = 1.0
MIN_FULLTEXT_WORDS = 500

USER_AGENT = "Mozilla/5.0 (compatible; DCJobPotentialComparatorRecovery/1.0; academic source audit)"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "text/html,application/pdf,*/*"}

review = pd.read_csv(REVIEW_TEMPLATE)
review = review.reset_index(drop=True)
review.insert(0, "priority", range(1, len(review) + 1))

print("Comparator sources to recover:", len(review))
review[["priority", "source_id", "asset_type", "project_or_source_name", "source_url"]]

In [ ]:
def is_missing(value) -> bool:
    return value is None or (isinstance(value, float) and math.isnan(value)) or not str(value).strip()


def safe_filename(name: str, default="source") -> str:
    name = str(name or default)
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("_")
    return name[:170] or default


def short_slug(value: str, limit=55) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value or "source")).strip("_")
    return value[:limit] or "source"


def normalise_url(value: str) -> str:
    if is_missing(value):
        return ""
    url = str(value).strip()
    if url.lower().startswith("doi:"):
        url = url[4:].strip()
    if re.match(r"^10\.\d{4,9}/", url, flags=re.I):
        return "https://doi.org/" + url
    if not re.match(r"^https?://", url, flags=re.I) and "." in url:
        return "https://" + url
    return url


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def classify_bytes(data: bytes, url="") -> dict:
    head = data[:4096]
    lower = head.decode("utf-8", errors="ignore").lower()
    if data.startswith(b"%PDF-"):
        return {"content_class": "pdf", "is_extractable": True, "block_reason": ""}
    if "just a moment" in lower or "cloudflare" in lower or "enable javascript" in lower:
        return {"content_class": "html_challenge", "is_extractable": False, "block_reason": "javascript_or_cloudflare_challenge"}
    if b"<html" in head.lower() or b"<!doctype html" in head.lower():
        return {"content_class": "html", "is_extractable": True, "block_reason": ""}
    if len(data) == 0:
        return {"content_class": "empty", "is_extractable": False, "block_reason": "empty_file"}
    try:
        data[:2000].decode("utf-8")
        return {"content_class": "text", "is_extractable": True, "block_reason": ""}
    except Exception:
        return {"content_class": "unknown_binary", "is_extractable": False, "block_reason": "unknown_format"}


def classify_path(path: Path, url="") -> dict:
    if not path.exists():
        return {"content_class": "missing", "is_extractable": False, "block_reason": "file_missing", "bytes": 0, "sha256": ""}
    data = path.read_bytes()
    out = classify_bytes(data, url=url)
    out.update({"bytes": len(data), "sha256": sha256_bytes(data) if data else ""})
    return out


def likely_landing_page(text: str, url="") -> bool:
    low = text.lower()
    doiish = "doi.org" in (url or "").lower()
    landing_terms = ["abstract", "purchase", "rent this article", "access through your institution", "sign in", "log in", "get access", "download pdf"]
    if doiish and any(t in low for t in landing_terms) and len(re.findall(r"\w+", text)) < 2500:
        return True
    return False


def find_pdf_links(html: str, base_url: str) -> list[str]:
    if BeautifulSoup is None:
        return []
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for key in ["citation_pdf_url", "bepress_citation_pdf_url", "dc.identifier"]:
        for tag in soup.find_all("meta"):
            name = (tag.get("name") or tag.get("property") or "").lower()
            content = tag.get("content") or ""
            if key in name and ".pdf" in content.lower():
                links.append(urljoin(base_url, content))
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        label = a.get_text(" ", strip=True).lower()
        blob = f"{href} {label}".lower()
        if ".pdf" in blob or "download pdf" in blob or label == "pdf":
            links.append(urljoin(base_url, href))
    seen = []
    for link in links:
        if link and link not in seen:
            seen.append(link)
    return seen[:5]


def download_url(url: str) -> tuple[bytes, str, str]:
    r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    r.raise_for_status()
    return r.content, r.headers.get("content-type", ""), r.url


print("Helpers ready.")

In [ ]:
# Recover/download sources. Existing manually saved files are always preferred.
status_rows = []

for _, src in review.iterrows():
    sid = safe_filename(src["source_id"])
    title = src.get("project_or_source_name", "")
    url = normalise_url(src.get("source_url", ""))
    base_name = safe_filename(f"{sid}_{short_slug(title)}")
    pdf_path = RAW_DIR / f"{base_name}.pdf"
    html_path = RAW_DIR / f"{base_name}.html"
    txt_path = RAW_DIR / f"{base_name}.txt"

    status = {
        "priority": src.get("priority", ""),
        "source_id": sid,
        "asset_type": src.get("asset_type", ""),
        "title": title,
        "url": url,
        "expected_pdf_file": pdf_path.name,
        "expected_html_file": html_path.name,
        "raw_path": "",
        "retrieval_status": "not_attempted",
        "content_type_header": "",
        "final_url": "",
        "content_class": "missing",
        "is_extractable": False,
        "block_reason": "",
        "bytes": 0,
        "sha256": "",
        "notes": src.get("metric_notes", ""),
    }

    existing = next((p for p in [pdf_path, html_path, txt_path] if p.exists()), None)
    if existing:
        status["raw_path"] = str(existing)
        status.update(classify_path(existing, url=url))
        status["retrieval_status"] = "already_in_comparator_raw_sources"
    elif not url:
        status["retrieval_status"] = "manual_required_no_url"
        status["block_reason"] = "no_url_in_review_template"
    elif not RUN_DOWNLOADS:
        status["retrieval_status"] = "download_disabled"
        status["block_reason"] = "RUN_DOWNLOADS_false"
    else:
        try:
            data, ctype, final_url = download_url(url)
            first_class = classify_bytes(data, url=final_url)
            status["content_type_header"] = ctype
            status["final_url"] = final_url

            if first_class["content_class"] == "pdf":
                pdf_path.write_bytes(data)
                status["raw_path"] = str(pdf_path)
                status.update(classify_path(pdf_path, url=final_url))
                status["retrieval_status"] = "downloaded_pdf"
            elif first_class["content_class"] == "html":
                html = data.decode("utf-8", errors="ignore")
                pdf_links = find_pdf_links(html, final_url)
                downloaded_pdf = False
                for pdf_url in pdf_links:
                    try:
                        pdf_data, pdf_ctype, pdf_final = download_url(pdf_url)
                        if classify_bytes(pdf_data, url=pdf_final)["content_class"] == "pdf":
                            pdf_path.write_bytes(pdf_data)
                            status["raw_path"] = str(pdf_path)
                            status["content_type_header"] = pdf_ctype
                            status["final_url"] = pdf_final
                            status.update(classify_path(pdf_path, url=pdf_final))
                            status["retrieval_status"] = "downloaded_pdf_from_html_link"
                            downloaded_pdf = True
                            break
                    except Exception:
                        continue
                if not downloaded_pdf:
                    html_path.write_bytes(data)
                    status["raw_path"] = str(html_path)
                    status.update(classify_path(html_path, url=final_url))
                    status["retrieval_status"] = "downloaded_html_or_landing_page"
            else:
                txt_path.write_bytes(data)
                status["raw_path"] = str(txt_path)
                status.update(classify_path(txt_path, url=final_url))
                status["retrieval_status"] = "downloaded_non_pdf_non_html"
        except Exception as e:
            status["retrieval_status"] = "download_failed"
            status["block_reason"] = f"{type(e).__name__}: {e}"
        time.sleep(SLEEP_BETWEEN_DOWNLOADS)

    status_rows.append(status)

status_df = pd.DataFrame(status_rows)
status_path = REVIEW_DIR / "comparator_source_recovery_status.csv"
status_df.to_csv(status_path, index=False)
print("Wrote", status_path)
status_df[["priority", "source_id", "retrieval_status", "content_class", "is_extractable", "block_reason"]]

In [ ]:
def extract_pdf_text(path: Path) -> str:
    pages = []
    try:
        import pdfplumber
        with pdfplumber.open(str(path)) as pdf:
            for i, page in enumerate(pdf.pages, 1):
                pages.append(f"[[PAGE {i}]]\n" + (page.extract_text() or ""))
        return "\n\n".join(pages)
    except Exception as e1:
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            for i, page in enumerate(reader.pages, 1):
                pages.append(f"[[PAGE {i}]]\n" + (page.extract_text() or ""))
            return "\n\n".join(pages)
        except Exception as e2:
            return f"[[EXTRACTION_ERROR]] pdfplumber={e1}; pypdf={e2}"


def html_to_text(path: Path) -> str:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if BeautifulSoup is None:
        return re.sub(r"<[^>]+>", " ", raw)
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return soup.get_text("\n", strip=True)


text_rows = []
for _, r in status_df.iterrows():
    raw_path = Path(str(r.get("raw_path", "")))
    if not bool(r.get("is_extractable")) or not raw_path.exists():
        continue
    cls = r.get("content_class", "")
    if cls == "pdf":
        text = extract_pdf_text(raw_path)
    elif cls in {"html", "text"}:
        text = html_to_text(raw_path) if cls == "html" else raw_path.read_text(encoding="utf-8", errors="ignore")
    else:
        continue

    words = len(re.findall(r"\w+", text))
    landing = likely_landing_page(text, url=r.get("url", ""))
    out = TEXT_DIR / f"{safe_filename(r['source_id'])}.txt"
    out.write_text(text, encoding="utf-8")
    text_rows.append({
        "priority": r.get("priority", ""),
        "source_id": r["source_id"],
        "title": r.get("title", ""),
        "text_file": str(out),
        "content_class": cls,
        "words": words,
        "likely_landing_page_or_abstract": landing,
        "source_raw_path": str(raw_path),
        "usable_for_metric_extraction": words >= MIN_FULLTEXT_WORDS and not landing,
    })

text_df = pd.DataFrame(text_rows)
text_path = REVIEW_DIR / "comparator_fulltext_inventory.csv"
text_df.to_csv(text_path, index=False)
print("Wrote", text_path)
text_df.sort_values(["usable_for_metric_extraction", "words"], ascending=[True, False]) if not text_df.empty else "No text extracted"

In [ ]:
usable = set(text_df.loc[text_df["usable_for_metric_extraction"], "source_id"]) if not text_df.empty else set()
landing_ids = set(text_df.loc[text_df.get("likely_landing_page_or_abstract", pd.Series(dtype=bool)).fillna(False), "source_id"]) if not text_df.empty else set()

queue = status_df[~status_df["source_id"].isin(usable)].copy()

def instruction(row):
    sid = row["source_id"]
    if sid in landing_ids:
        return "The download appears to be a DOI/publisher landing page or abstract. Download the full PDF through browser/library and save it using expected_pdf_file, then rerun this notebook."
    if row.get("retrieval_status") == "manual_required_no_url":
        return "Find the official source manually, download PDF or save HTML, and use expected_pdf_file or expected_html_file."
    if row.get("content_class") in ["missing", "html_challenge"]:
        return "Download the full source manually through browser/library and save it using expected_pdf_file or expected_html_file."
    if row.get("retrieval_status") == "download_failed":
        return "Open the URL manually, download the PDF/full text, and save it using expected_pdf_file or expected_html_file."
    return "Check whether the captured page is sufficient. If not, download the full PDF/full text manually and rerun."

queue["ra_instruction"] = queue.apply(instruction, axis=1)
queue = queue[[
    "priority", "source_id", "title", "url", "expected_pdf_file", "expected_html_file",
    "retrieval_status", "content_class", "block_reason", "ra_instruction", "notes",
]].sort_values(["priority", "source_id"])

queue_path = QUEUE_DIR / "comparator_manual_download_queue.csv"
queue.to_csv(queue_path, index=False)

print("Wrote", queue_path)
print("Total comparator sources:", len(status_df))
print("Usable full texts:", len(usable))
print("Manual/check queue:", len(queue))
queue

## Manual Download Rule

If a source remains in `comparator_manual_download_queue.csv`, open its URL in the browser, download the real PDF/full text, and save it into:

`Research_Workflow/02_Source_Extraction/data/comparator_sources/raw_sources/`

Use the exact `expected_pdf_file` or `expected_html_file` shown in the queue. Then rerun this notebook. When the manual queue is small or empty, the comparator text can be mined for metrics.